# M.I.N.E.R.V.A. — Run Pipeline v2.0 (Colab)**Misinformation Investigation through Networked Embeddings for Rumor Verification and Awareness**This notebook runs the v2.0 refactored pipeline that fixes the four legacy issues:1. **Static explanations** — replaced by 56-entry response bank with 3-tier (novice / proficient / advanced) varying output.2. **Off-theme content leaks** — replaced by classifier+hard-negatives gate (script 23) with structured rejection log.3. **Random pseudonyms** — replaced by deterministic 3-candidate router (script 22).4. **Truncated GPT-2 generations** — caught by `is_truncated()` gate before they reach the deck.Pipeline order (script numbers preserved from legacy for reviewer continuity):```[GPT-2 generation] → 13 → 18 → 21 → 22 → 23 → 24 → 25 → 26 → 27```References (full citations in `docs/MASTER_CODEBOOK.md`):Roozenbeek & van der Linden (2019); Khosravi et al. (2022); Longo et al. (2024); Liu, Ye & Li (2024); Athira et al. (2023); Christensen et al. (2022); Brolós et al. (2021); Caulfield (2019); Modirrousta-Galian & Higham (2023); Arugay & Baquisal (2022); Schipper (2025).

## 1. Mount Drive and clone repoIf you already have the v1 notebook setup, this is the same. The new files live in `scripts/`, `templates/`, `tests/`, `docs/`, `notebooks/`.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')PROJECT_ROOT = '/content/drive/MyDrive/MINERVA'  # adjust if needed%cd $PROJECT_ROOT# Pull the upgrade branch (recommended)# !git pull origin upgrade/minerva-election-themeimport os, syssys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))print('PROJECT_ROOT:', PROJECT_ROOT)

## 2. Install dependencies`pydantic >= 2.0` is the only NEW dependency added in v2.0.

In [ ]:
!pip install -q pydantic pytest# Existing legacy deps below — uncomment if running fresh# !pip install -q transformers torch sentence-transformers feyn pandas scikit-learn

## 3. Sanity check: run all unit testsIf any of these fail, the rest of the pipeline output cannot be trusted.

In [ ]:
!python -m pytest tests/ -v

## 4. Inspect the response bank statsThe bank is the explanation engine. Verify version + hash so you can A/B-compare deck batches later.

In [ ]:
from minerva_response_bank import bank_stats, BANK_VERSIONimport jsonstats = bank_stats()print(json.dumps(stats, indent=2))

## 5. Inspect the candidate registryThree fictional candidates — `C-RM` Marquez (DYNASTIC), `C-IB` Bantayan (REFORMIST), `C-JS` Salonga (POPULIST) — backed by Filipino electoral-disinformation archetypes (Arugay & Baquisal 2022; Schipper 2025).

In [ ]:
from minerva_candidates import REGISTRYfor code, c in REGISTRY.items():    print(f"\n=== {code} — {c.name} ({c.archetype}) ===")    print(f"  Region: {c.region}")    print(f"  Party:  {c.party_name} ({c.party_acronym})")    print(f"  Slogan: {c.platform_slogan}")    print(f"  Top susceptible indicators:",          sorted(c.indicator_weights.items(), key=lambda kv: -kv[1])[:3])

## 6. (If needed) Run scripts 1–12 from legacy v1 notebookScripts 1 through 12 (data ingest, RoBERTa/DistilBERT/DE-GNN training, GPT-2 fine-tune, embedding extraction) are unchanged in v2.0. If you have already produced `generated/gpt2_synthetic_raw_*.jsonl` from the legacy notebook, **skip this section**.```python!python scripts/01_load_jcblaise.py ...!python scripts/02_train_roberta_filipino.py ...# ... legacy scripts 03–12 unchanged ...```

## 7. Script 13 — Score generated posts (REFACTORED)Scores GPT-2 outputs with QLattice + named features + truncation gate.**Bug fix from legacy notebook**: the legacy cell 23 was monkey-patching script 12 to work around a tokenizer issue. This refactor's `13_score_generated_with_qlattice.py` doesn't need that patch.

In [ ]:
!python scripts/13_score_generated_with_qlattice.py \    --in_file  generated/gpt2_synthetic_raw_both.jsonl \    --out_file generated/gpt2_synthetic_final_both.jsonl \    --qlattice_model models/qlattice_model.pkl \    --pca_models    models/pca_models.pkl \    --rejection_log reports/score_rejection_log.jsonl

## 8. Script 18 — Verdict + explanation (REFACTORED)This is where the static-explanation bug is fixed. Each card now gets:- A **fired_indicators** list (from the 12-cue taxonomy).- A **content-aware summary** drawn from the response bank.- A **bank_ref** per phrase (e.g. `EMO/v1.0/n3`) for audit + A/B testing.After this cell runs, the diversity log line should read close to **100%** unique summaries — proving the static-explanation fix.

In [ ]:
!python scripts/18_verdict_explain.py \    --in_file   generated/gpt2_synthetic_final_both.jsonl \    --out_file  generated/unity_cards.json \    --audit_out reports/audit_18.jsonl \    --seed 1729

## 9. Script 21 — Balance unity cards (REFACTORED)Validates against `UnityCard` schema, then balances across verdict × candidate × difficulty × indicator coverage.

In [ ]:
!python scripts/21_balance_unity_cards.py \    --in_file  generated/unity_cards.json \    --out_file generated/unity_cards_balanced.json \    --target_total 200 \    --report_out reports/balance_report.json

## 10. Script 22 — Pseudonymise (REFACTORED)Maps any detected real-name reference to one of the three fictional candidates with **session-cache consistency** (same input → same output, throughout the run).

In [ ]:
!python scripts/22_pseudonymize_entities.py \    --in_file  generated/unity_cards_balanced.json \    --out_file generated/unity_cards_pseudonymized.json \    --re_explain \    --seed 1729

## 11. Script 23 — Enforce electoral theme (REFACTORED)Two-stage filter: keyword baseline + DistilBERT classifier head. Hard-negative awareness (transport, utilities, sports, K-pop) prevents the legacy Grab/Meralco leaks.Use `--strict_no_neutral` to disable the neutral-volume pass-through if you want a pure-electoral feed.

In [ ]:
!python scripts/23_enforce_election_theme.py \    --in_file  generated/unity_cards_pseudonymized.json \    --out_file generated/unity_cards_themed.json \    --theme_threshold 0.55 \    --report_out reports/theme_filter_report.json \    --rejection_log reports/theme_rejection_log.jsonl

## 12. Script 24 — Curate teaching cards (REFACTORED)Promotes themed cards into the daily-cycle deck with:- Difficulty banding (novice → proficient → advanced).- Mandatory ≥ 3 credible cards per day (Modirrousta-Galian & Higham 2023 mandate).- Cross-link each FAKE to the most-recent REAL the player saw, so VERIdict can show side-by-side comparison.

In [ ]:
!python scripts/24_curate_teaching_cards.py \    --in_file  generated/unity_cards_themed.json \    --out_file generated/story_cards.json \    --reject_out generated/story_cards_rejected.json \    --report_out reports/story_cards_curation_report.json \    --days 7 --cards_per_day 8 --min_credible_per_day 3 \    --seed 1729

## 13. Script 25 — Build candidate scenarios (REFACTORED)VERIdex profile cards: archetype-grounded biography, platform planks, indicator-susceptibility heatmap (prior + empirical), counter-narrative anchors.

In [ ]:
!python scripts/25_build_candidate_scenarios.py \    --story_cards generated/story_cards.json \    --out_file    generated/candidate_scenarios.json

## 14. Script 26 — Faithfulness audit (NEW)Re-extracts indicators from every explanation prose string and asserts set-equality with `fired_indicators`. **Must pass with 100%** before a deck ships to students.If the strict mode exits non-zero, inspect `reports/faithfulness_failures.jsonl`.

In [ ]:
!python scripts/26_faithfulness_audit.py \    --in_file       generated/story_cards.json \    --report_out    reports/faithfulness_audit_report.json \    --failures_out  reports/faithfulness_failures.jsonl# Add --strict to abort on any failure

## 15. Script 27 — Bank versioning (NEW)Stamp the final deck with the bank version + hash so future A/B comparisons are reproducible.

In [ ]:
!python scripts/27_response_bank_versioning.py stamp \    --in_file  generated/story_cards.json \    --out_file generated/story_cards_stamped.json!python scripts/27_response_bank_versioning.py export \    --out_file templates/teaching_response_bank_v1_export.json

## 16. Final reports dashboardPrint all the v2.0-specific reports in one place for the thesis paper appendix.

In [ ]:
import json, globfor fp in sorted(glob.glob('reports/*.json')):    print(f'\n========================')    print(f'{fp}')    print(f'========================')    print(json.dumps(json.load(open(fp)), indent=2)[:2000])

## 17. (Optional) Inspect a few sample cards from the final deck

In [ ]:
import jsondeck = json.load(open('generated/story_cards.json'))print(f'Deck size: {len(deck)} cards across {max(c["day"] for c in deck)} days')# Sample one card per (verdict, candidate) comboseen = set()for c in deck:    key = (c['verdict'], c['candidate'])    if key in seen: continue    seen.add(key)    print(f"\n--- Day {c['day']} | {c['verdict']} | {c['candidate']} | tier={c['explanation']['tier']} ---")    print(f"Text: {c['text'][:150]}...")    print(f"Fired: {c['fired_indicators']}")    print(f"Summary: {c['explanation']['summary'][:300]}")

---## Appendix A — Bug fixes from the legacy notebookThe legacy `MINERVA_Run_Revised_Colab.ipynb` had these issues:| Cell | Issue | v2.0 fix || ---- | ----- | -------- || 2    | Stray `e` token at top of file (syntax error after fresh checkout) | Removed in v2 notebook || 23   | Monkey-patched script 12 to bypass a tokenizer assertion | Not needed — script 13 in v2 catches truncation properly || 27   | Wrote unity_cards.json without schema validation | All output now passes through `UnityCard.model_validate` |---## Appendix B — Where to look if a stage fails| Stage     | If it fails, check    || --------- | --------------------- || Tests     | `pytest tests/` output — fix detector before continuing || 13        | `reports/score_rejection_log.jsonl` for truncation reasons || 18        | `reports/audit_18.jsonl` for schema-validation reasons || 21        | `reports/balance_report.json` for distribution skew || 22        | Card-level `metadata.pseudonym_replacements` for what got renamed || 23        | `reports/theme_rejection_log.jsonl` per-card rejection reasons || 24        | `generated/story_cards_rejected.json` || 26        | `reports/faithfulness_failures.jsonl` |